# Ensemble Methods for Credit Risk — Probability of Default (PD) Modelling

**Dataset**: UCI *Default of Credit Card Clients* — Yeh & Lien (2009)  
**Target**: `DEFAULT_NEXT_MONTH` — binary: 1 = default, 0 = no default  
**Class balance**: ~22% default (minority class) — all models must handle this explicitly  

---

This notebook demonstrates four core ensemble techniques:
1. **Bagging** — parallel aggregation of models trained on random subsets
2. **Boosting** — sequential error correction (XGBoost, Gradient Boosting)
3. **Stacking** — meta-learning across diverse base models
4. **Voting** — hard and soft vote aggregation

Each section explains what the method is, why it is useful in ML, and how it benefits credit risk prediction specifically.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, BaggingClassifier,
    GradientBoostingClassifier, StackingClassifier, VotingClassifier
)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score,
    roc_curve, log_loss, classification_report,
    precision_score, recall_score
)
import matplotlib.pyplot as plt
import sys
print(sys.executable)

pd.set_option("display.max_columns", None)

c:\Users\Kasia\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
# ── paths ──────────────────────────────────────────────────────────────────
REPO_ROOT  = Path(r"F:\Apps\Credit-Risk-Score-ML")
DATA_PATH  = REPO_ROOT / "data" / "raw" / "default of credit card clients.xls"
RANDOM_STATE = 42

print("Data path:", DATA_PATH)
print("Exists:   ", DATA_PATH.exists())

Data path: F:\Apps\Credit-Risk-Score-ML\data\raw\default of credit card clients.xls
Exists:    True


In [3]:
df_raw = pd.read_excel(DATA_PATH, sheet_name=0, engine="xlrd", header=1)
df = df_raw.rename(columns={"default payment next month": "DEFAULT_NEXT_MONTH"})

# fix undocumented category codes
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"]  = df["MARRIAGE"].replace({0: 3})

print("Shape:", df.shape)
print("\nTarget distribution (%):")
print((df["DEFAULT_NEXT_MONTH"].value_counts(normalize=True) * 100).round(2))

Shape: (30000, 25)

Target distribution (%):
DEFAULT_NEXT_MONTH
0    77.88
1    22.12
Name: proportion, dtype: float64


In [4]:
# ── features & target ───────────────────────────────────────────────────────
TARGET = "DEFAULT_NEXT_MONTH"

# Drop ID — not a predictive feature
X = df.drop(columns=["ID", TARGET], errors="ignore")
y = df[TARGET].astype(int)

# One-hot encode the three categorical columns
X = pd.get_dummies(X, columns=["SEX", "EDUCATION", "MARRIAGE"], drop_first=True)

# Train / test split — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale — important for KNN, SVM, Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Class imbalance weight for XGBoost (count negatives / count positives)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Train:", X_train.shape, "  Test:", X_test.shape)
print("Default rate — train:", y_train.mean().round(4),
      "  test:", y_test.mean().round(4))
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

Train: (24000, 26)   Test: (6000, 26)
Default rate — train: 0.2212   test: 0.2212
scale_pos_weight: 3.52


In [ ]:
# ── evaluation helper ───────────────────────────────────────────────────────
# Matches signature from hyperparameter notebook.
# PR-AUC added alongside ROC-AUC — more informative for imbalanced credit data
# (Saito & Rehmsmeier, 2015). A naive all-negative classifier gets 78% accuracy
# but near-zero PR-AUC, making it easy to spot poor models.

results = {}

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name, scaled=False, fit=True):
    if fit:
        model.fit(X_tr, y_tr)
    preds  = model.predict(X_te)
    probs  = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    acc  = accuracy_score(y_te, preds)
    f1   = f1_score(y_te, preds, average="macro")
    auc  = roc_auc_score(y_te, probs)           if probs is not None else None
    pr   = average_precision_score(y_te, probs) if probs is not None else None
    ll   = log_loss(y_te, probs)                if probs is not None else None

    results[name] = {"ROC-AUC": auc, "PR-AUC": pr, "Macro F1": f1,
                     "Accuracy": acc, "Log Loss": ll}

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Macro-F1  : {f1:.4f}")
    if auc: print(f"  ROC-AUC   : {auc:.4f}")
    if pr:  print(f"  PR-AUC    : {pr:.4f}")
    if ll:  print(f"  Log Loss  : {ll:.4f}")
    print(classification_report(y_te, preds, target_names=["No Default", "Default"]))
    return model

---
# 1. Bagging

## What is Bagging?

**Bagging** (Bootstrap Aggregating) trains multiple copies of the same base model on different random subsets of the training data (sampled *with replacement* — called bootstrapping). Final predictions are made by majority vote (classification) or averaging (regression).

## Why use it in ML?

- Reduces **variance** without significantly increasing bias.
- Works best with high-variance, low-bias base models (e.g. deep decision trees, KNN).
- Each bootstrap sample is ~63% of the original data — the remaining 37% (out-of-bag samples) can be used for internal validation at no extra cost.
- Parallelisable — all base models are trained independently.

## How does it benefit credit risk prediction?

- KNN is sensitive to noisy credit features (BILL_AMT outliers, irregular payment sequences). Bagging smooths out this sensitivity across 30 bootstrapped samples.
- The UCI dataset has complex non-linear interactions between PAY_X, BILL_AMT, and LIMIT_BAL — high-variance models capture these but overfit individually. Bagging stabilises them.
- In regulatory context, a stable model is easier to validate and document (Basel II Model Risk Management requirements).

## Requirements

- Base estimator must implement `.fit()` and `.predict_proba()`.
- `n_estimators`: typically 20–100. More = more stable, diminishing returns beyond ~50.
- Feature **scaling required** when base model is distance-based (KNN) — use `X_train_sc` / `X_test_sc`.

In [ ]:
base_knn = KNeighborsClassifier(n_neighbors=5)

bagging_knn = BaggingClassifier(
    estimator=base_knn,
    n_estimators=30,
    random_state=RANDOM_STATE
)

# KNN needs scaled data
bagging_knn = evaluate_model(bagging_knn, X_train_sc, y_train, X_test_sc, y_test,
                              "Bagging — KNN (scaled)")

In [ ]:
# Bagging with Logistic Regression — linear base model, scaling required
# class_weight='balanced' compensates for the 22%/78% default/non-default split
base_lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)

bagging_lr = BaggingClassifier(
    estimator=base_lr,
    n_estimators=30,
    random_state=RANDOM_STATE
)

bagging_lr = evaluate_model(bagging_lr, X_train_sc, y_train, X_test_sc, y_test,
                             "Bagging — Logistic Regression (scaled)")

---
# 2. Boosting

## What is Boosting?

**Boosting** trains models **sequentially**. Each new model focuses on the mistakes of the previous one — incorrectly classified samples receive higher weights so the next learner pays more attention to them. Final predictions combine all models weighted by their individual accuracy.

Key algorithms used here:
- **XGBoost** — regularised gradient boosting with column subsampling and `scale_pos_weight` for class imbalance. Faster than sklearn GB. Native SHAP support.
- **Gradient Boosting (sklearn)** — classic implementation, robust and reliable baseline. No built-in imbalance parameter; compensated via `sample_weight`.

## Why use it in ML?

- Reduces **bias** — converts weak learners (shallow trees) into a strong ensemble.
- Generally outperforms Bagging on structured tabular data.
- Built-in feature importance and native SHAP compatibility.

## How does it benefit credit risk prediction?

- Captures complex non-linear patterns in PAY_X repayment sequences and LIMIT_BAL interactions that linear models miss.
- `scale_pos_weight` directly addresses the 22%/78% imbalance without oversampling — important for preserving real-world distribution.
- XGBoost + SHAP satisfies EU AI Act transparency requirements: each loan decision can be explained with a waterfall chart showing which features increased or decreased PD.
- In Lessmann et al. (2015) benchmark, gradient boosting variants consistently outperformed logistic regression on credit scoring tasks.

## Requirements

- `n_estimators`: number of trees. Use cross-validation or early stopping to find optimal value.
- `learning_rate`: lower = more trees needed but better generalisation. Typical range: 0.01–0.3.
- `scale_pos_weight` (XGBoost): set to `count(negative) / count(positive)` ≈ 3.5 for this dataset.
- Feature scaling **not required** for tree-based boosting.

In [ ]:
# XGBoost — regularised gradient boosting
# scale_pos_weight handles the 22%/78% class imbalance (~3.5x more non-defaulters)
# No scaling needed — tree-based model uses raw feature values
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    use_label_encoder=False
)

xgb_model = evaluate_model(xgb_model, X_train, y_train, X_test, y_test, "XGBoost")

In [ ]:
# Gradient Boosting (sklearn) — no scale_pos_weight parameter
# Compensate for class imbalance via sample_weight computed from class distribution
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=RANDOM_STATE
)

gb_model.fit(X_train, y_train, sample_weight=sample_weights)
gb_model = evaluate_model(gb_model, X_train, y_train, X_test, y_test,
                           "Gradient Boosting", fit=False)

---
# 3. Stacking

## What is Stacking?

**Stacking** (Stacked Generalisation) trains multiple diverse **base models** (level-0), then uses their predictions as input features to train a **meta-model** (level-1) that learns the optimal way to combine them.

Key difference from Voting: the meta-model **learns** how to weight each base model rather than using a fixed rule. Scikit-learn uses cross-validation internally to generate base model predictions, preventing data leakage.

## Why use it in ML?

- Captures complementary strengths of different model families.
- The meta-model automatically learns to trust different base models on different input regions.
- Often achieves higher performance than any single base model alone.

## How does it benefit credit risk prediction?

- Credit default is driven by diverse signals: recent payment behaviour (best captured by tree models), long-term balance trends (linear models), and complex interaction effects (SVM). Stacking integrates all of them.
- Logistic Regression as meta-model preserves interpretability at the ensemble level — regulators can inspect the meta-coefficients to understand how base models are combined.
- In thesis context: stacking result vs best single model (XGBoost) quantifies the gain from ensemble integration.

## Requirements

- Base models should be diverse — combining similar models gives little extra benefit.
- All base models must support `predict_proba` if meta-model uses probability outputs.
- `cv=5` controls cross-validation folds for generating level-0 predictions.
- Computationally expensive: 5 folds × number of base models × training cost.
- Scale-sensitive models (SVM, LR) are wrapped in pipelines so stacking handles scaling internally.

In [ ]:
# Wrap scale-sensitive models in pipelines — stacking calls each model independently
base_models = [
    ("rf",  RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                    random_state=RANDOM_STATE)),
    ("xgb", xgb.XGBClassifier(n_estimators=100, scale_pos_weight=scale_pos_weight,
                                eval_metric="logloss", random_state=RANDOM_STATE,
                                use_label_encoder=False)),
    ("svm", make_pipeline(StandardScaler(),
                           SVC(probability=True, class_weight="balanced",
                               random_state=RANDOM_STATE)))
]

# Meta-model: LR — interpretable combination layer
meta_model = LogisticRegression(class_weight="balanced", max_iter=1000,
                                 random_state=RANDOM_STATE)

stack_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    passthrough=False
)

# Unscaled X — pipelines inside handle scaling per base model
stack_clf = evaluate_model(stack_clf, X_train, y_train, X_test, y_test,
                            "Stacking (RF + XGB + SVM → LR)")

---
# 4. Voting Ensemble

## What is a Voting Ensemble?

A **Voting Classifier** trains multiple models independently (same as Bagging, but with different model types) and combines predictions:

- **Hard Voting** — each model votes for one class; majority wins. No probabilities needed.
- **Soft Voting** — each model outputs class probabilities; these are averaged. The class with the highest average probability wins. Generally more accurate than hard voting when models are well-calibrated.

## Why use it in ML?

- Simple to implement and easy to explain to non-technical stakeholders.
- Reduces variance when models make different types of errors.
- Soft voting produces a smooth probability score — usable as a credit risk score.

## How does it benefit credit risk prediction?

- Combining interpretable model (LR) with high-performance models (RF, XGBoost) lets you satisfy both regulatory interpretability requirements and performance goals simultaneously.
- Soft voting output is calibrated PD score suitable for Expected Loss: `EL = PD × LGD × EAD`.
- Custom decision thresholds can be applied post-scoring — e.g. flag top 10% highest-risk borrowers for manual review.

## Requirements

- Hard voting: only `.predict()` needed.
- Soft voting: all base models must support `.predict_proba()`. SVC needs `probability=True`.
- Scale-sensitive models wrapped in pipelines so VotingClassifier handles them correctly.

In [ ]:
clf_lr  = make_pipeline(StandardScaler(),
                         LogisticRegression(class_weight="balanced", max_iter=1000,
                                            random_state=RANDOM_STATE))
clf_rf  = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                  random_state=RANDOM_STATE)
clf_xgb = xgb.XGBClassifier(n_estimators=100, scale_pos_weight=scale_pos_weight,
                              eval_metric="logloss", random_state=RANDOM_STATE,
                              use_label_encoder=False)

# Hard Voting: majority class vote — no probabilities involved
hard_voter = VotingClassifier(
    estimators=[("lr", clf_lr), ("rf", clf_rf), ("xgb", clf_xgb)],
    voting="hard"
)

hard_voter = evaluate_model(hard_voter, X_train, y_train, X_test, y_test,
                             "Hard Voting (LR + RF + XGB)")

In [ ]:
clf_lr2  = make_pipeline(StandardScaler(),
                          LogisticRegression(class_weight="balanced", max_iter=1000,
                                             random_state=RANDOM_STATE))
clf_rf2  = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                   random_state=RANDOM_STATE)
clf_xgb2 = xgb.XGBClassifier(n_estimators=100, scale_pos_weight=scale_pos_weight,
                               eval_metric="logloss", random_state=RANDOM_STATE,
                               use_label_encoder=False)

# Soft Voting: average predicted probabilities — more nuanced than hard voting
soft_voter = VotingClassifier(
    estimators=[("lr", clf_lr2), ("rf", clf_rf2), ("xgb", clf_xgb2)],
    voting="soft"
)

soft_voter = evaluate_model(soft_voter, X_train, y_train, X_test, y_test,
                             "Soft Voting (LR + RF + XGB)")

---
# 5. Log Loss

## What is Log Loss?

**Log Loss** (Binary Cross-Entropy) measures quality of predicted **probabilities**, not just labels. It penalises confident wrong predictions heavily — a model that assigns 99% probability of no-default when the borrower actually defaults receives very high (bad) log loss.

`L = -1/N × Σ [y·log(p) + (1-y)·log(1-p)]`  →  lower is better.

## Why use it in ML?

- Directly optimised by LR and XGBoost (`eval_metric='logloss'`).
- More informative than accuracy for probability-producing models.
- Essential when downstream decisions depend on the probability score, not just binary label.

## How does it benefit credit risk prediction?

- Credit decisioning uses the **probability of default** to set interest rates, credit limits, and reserve requirements (Basel II IRB approach).
- A well-calibrated model (low log loss) feeds reliable PD estimates into Expected Loss: `EL = PD × LGD × EAD`.
- Log loss complements ROC-AUC: model can rank defaulters well (high AUC) but produce unreliable probabilities (high log loss). Both matter in credit risk.

In [ ]:
print(f"\n{'='*55}")
print("  Log Loss — All Ensemble Models")
print(f"{'='*55}")
print(f"{'Model':<45} {'Log Loss':>10}")
print("-" * 57)
for name, m in sorted(results.items(), key=lambda x: x[1]["Log Loss"] or 99):
    ll = m["Log Loss"]
    print(f"{name:<45} {ll:>10.4f}" if ll else f"{name:<45} {'N/A':>10}")

---
# 6. ROC-AUC Curve

## What is ROC-AUC?

The **ROC curve** plots **True Positive Rate** (recall for defaulters) against **False Positive Rate** (false alarm rate for non-defaulters) at every classification threshold.

**AUC** summarises the curve:
- AUC = 1.0 → perfect classifier
- AUC = 0.5 → random (diagonal line)
- AUC 0.70–0.80 → acceptable for credit risk
- AUC > 0.80 → good discriminatory power

## Why use it in ML?

- Threshold-independent — evaluates across all operating points.
- Industry standard for binary classifier comparison.
- More informative than accuracy on imbalanced datasets.

## How does it benefit credit risk prediction?

- ROC-AUC is the primary metric required for PD model validation under Basel II and EBA RTS on the IRB approach.
- Allows risk managers to choose threshold based on business cost of false negatives (missed defaults) vs false positives (rejected good borrowers).
- Multi-model ROC plot communicates performance comparison to non-technical stakeholders clearly.

In [ ]:
models_to_plot = {
    "Bagging — KNN":          (bagging_knn, X_test_sc),
    "Bagging — LR":           (bagging_lr,  X_test_sc),
    "XGBoost":                (xgb_model,   X_test),
    "Gradient Boosting":      (gb_model,    X_test),
    "Stacking":               (stack_clf,   X_test),
    "Hard Voting":            (hard_voter,  X_test),
    "Soft Voting":            (soft_voter,  X_test),
}

plt.figure(figsize=(10, 7))

for name, (model, X_te) in models_to_plot.items():
    y_proba = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curves — Ensemble Models on Credit Risk Data")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

---
# 7. Results Summary

## How to interpret the comparison

When selecting best model for credit risk project, evaluate across four dimensions:

1. **Performance** — ROC-AUC and PR-AUC are primary. PR-AUC is especially important because it reflects performance on minority (default) class directly.
2. **Calibration** — Log Loss reflects reliability of probability estimates. Low log loss → PD scores usable in Expected Loss calculations.
3. **Interpretability** — Required under EU AI Act (high-risk AI) and Basel II model documentation. XGBoost + SHAP or Logistic Regression preferred.
4. **Stability** — Ensemble models (Bagging, Stacking) produce more stable scores across data samples — important for regulatory model validation.

In [ ]:
print(f"\n{'='*55}")
print("  Final Model Comparison — All Ensemble Methods")
print(f"{'='*55}")
summary_df = pd.DataFrame(results).T
summary_df = summary_df.sort_values("ROC-AUC", ascending=False)
print(summary_df.to_string(float_format="{:.4f}".format))

---
## Next Steps

1. **Hyperparameter tuning** — use `RandomizedSearchCV` or Bayesian optimisation (`skopt`) with `scoring='roc_auc'` to optimise XGBoost and Stacking (see `df_raw_ensemble_hyperparameter_credit_risk_project.ipynb`).
2. **SHAP explainability** — apply `shap.TreeExplainer` to best tree-based model for global feature importance and local prediction explanations. Required for EU AI Act Article 13 transparency.
3. **Threshold optimisation** — default 0.5 threshold is rarely optimal in credit risk. Plot precision-recall trade-off and select threshold based on business cost of False Negatives vs False Positives.
4. **Cross-validation** — replace single train/test split with stratified k-fold CV for stable performance estimates.
5. **Model governance** — document best model using Model Card format: intended use, limitations, performance metrics, monitoring plan (PSI drift, SHAP stability).